In [22]:
%reload_ext autoreload
%autoreload 2
from fun import *
import polars as pl
import pandas as pd
import datetime as dt
import time 

logging = get_logger(log_file='回测.log',inherit=False)

start_date = dt.date(2022,1,1)
end_date = dt.datetime.today()
end_date = dt.date(2024,1,1)
# 获取指定日期的日线数据
stock_data = read_day_data(start_date=start_date,end_date=end_date,file_path='ts_stock_all_data')
stock_data = stock_data.drop_nulls(subset=['open','close','pre_close','limit_up','limit_down'])
market_value = read_day_data(start_date=start_date,end_date=end_date,file_path='ts_daily_basic')
market_value = market_value.with_columns([
   ( pl.col('free_share')*pl.col('close')/1e4).alias('free_float_mv')
])
stock_data = stock_data.join(market_value.select(['code','trading_date','free_float_mv']),on=['code','trading_date'],how='left')
# stock_data.schema
# 去掉没用的列
stock_data = stock_data.drop(['change','total_share','attack','activity','pe','float_share','buying','selling','swing','strength','avg_turnover'])

In [23]:
# 1.涨停标记
# 标记涨停状态：limit_status
stock_data = mark_limit_status(stock_data)
# 标记涨停描述：limit_desc
stock_data = mark_limit_desc(stock_data)
# 记录最近的一次涨停描述：last_limit_desc
stock_data = mark_last_limit_desc(stock_data)
# 统计10天内涨停平均换手率
stock_data = cal_limit_avg_turnover(stock_data, window=5)

# 2.均线特征
# 计算均线:sma_{window}
stock_data_future = add_sma(stock_data, window=5)
stock_data_future = add_sma(stock_data_future, window=7)
stock_data_future = add_sma(stock_data_future, window=20)

# 3.计算开盘涨幅:open_chg,乖离率,成交均价vwap
stock_data_future = stock_data_future.with_columns(
    (
        (pl.col("open") - pl.col("pre_close")) 
        / pl.col("pre_close") 
        * 100
    ).alias("open_pct"),  # 开盘涨幅百分比
    #((pl.col("close") - pl.col("sma_7")) / pl.col("sma_7") * 100).alias("close_sma7_pct"), #乖离率
    (pl.col("amount")*100 / pl.col("volume")).alias("vwap"),
    ((pl.col("low") <= pl.col("limit_down")*1.01)).alias("touch_limit_down"), # 是否触及跌停
)

# 筛选stock_data行 为买入信号 1. 昨日涨停or断板or炸板   2.今日低开-3%至-4%  3.昨日收盘在昨日日五线上   4. 最近一次涨停描述 !=一天一板 或者 非空
# 4. 先确保数据按股票和日期排序
stock_data_future = stock_data_future.sort(["code", "trading_date"])

# 5. 筛选全市场热门股数据
热门股 = stock_data_future.filter(
    # 非st,创业,科创
    ~(pl.col("type").is_not_null() & (pl.col("type") == "ST")) &
    ~(pl.col("code").str.split(".").list[1].str.starts_with("30") | pl.col("code").str.split(".").list[1].str.starts_with("688")) &

    # 最近一次涨停描述
    (
        #(pl.col("last_limit_desc") != "1天1板") &
        (pl.col("last_limit_desc").is_not_null())
    ) 
)


In [24]:
# 利用每日热门股的平均pct变化率来判断市场热度
热门股_summary = 热门股.to_pandas().groupby("trading_date").agg({
    "open_pct": "mean",
    "free_float_mv": "mean",
    "pct": "mean",
    "code": "count"
}).rename(columns={
    "open_pct": "avg_open_pct",
    "free_float_mv": "avg_free_float_mv",
    "pct": "avg_pct",
    "code": "num_stocks"
}).reset_index().sort_values("trading_date")

# 柱状图，正值红色，负值绿色
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors_open = ["#d62728" if v > 0 else "#2ca02c" for v in 热门股_summary["avg_open_pct"]]
colors_mv = ["#d62728" if v > 0 else "#2ca02c" for v in 热门股_summary["avg_free_float_mv"]]
colors_pct = ["#d62728" if v > 0 else "#2ca02c" for v in 热门股_summary["avg_pct"]]
colors_num = ["#d62728" if v > 0 else "#2ca02c" for v in 热门股_summary["num_stocks"]]

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=("平均开盘涨幅 (%)", "平均自由流通市值 (亿元)", "平均涨跌幅 (%)", "热门股数量"))
fig.add_trace(go.Bar(x=热门股_summary['trading_date'], y=热门股_summary['avg_open_pct'], name='平均开盘涨幅 (%)', marker_color=colors_open), row=1, col=1)
fig.add_trace(go.Bar(x=热门股_summary['trading_date'], y=热门股_summary['avg_free_float_mv'], name='平均自由流通市值 (亿元)', marker_color=colors_mv), row=2, col=1)
fig.add_trace(go.Bar(x=热门股_summary['trading_date'], y=热门股_summary['avg_pct'], name='平均涨跌幅 (%)', marker_color=colors_pct), row=3, col=1)
fig.add_trace(go.Bar(x=热门股_summary['trading_date'], y=热门股_summary['num_stocks'], name='热门股数量', marker_color=colors_num), row=4, col=1)
fig.update_layout(height=900, width=800, title_text="全市场热门股每日特征趋势", showlegend=False)
fig.show()

In [25]:
# 利用avg_pct计算每日净值,单利息累积
热门股_summary['net_value'] = (1 + 热门股_summary['avg_pct'].cumsum() / 100)
#热门股_summary.set_index('trading_date')['net_value'].plot(title='全市场热门股每日净值曲线')

from stock_plot import *
# 设置时间为index value为net_value
net_curve = 热门股_summary.set_index('trading_date')['net_value'].to_frame()
mv_bar = 热门股_summary.set_index('trading_date')['avg_free_float_mv'].to_frame()
pct_bar = 热门股_summary.set_index('trading_date')['avg_pct'].to_frame()
num_bar = 热门股_summary.set_index('trading_date')['num_stocks'].to_frame()

# 画图
plot_interactive_curve(net_curve, title='全市场热门股每日净值曲线')
plot_interactive_bar_chart(mv_bar, title='平均自由流通市值 (亿元)')
plot_interactive_bar_chart(pct_bar, title='平均涨跌幅 (%)')
plot_interactive_bar_chart(num_bar, title='热门股数量')
                       
                        
